In [ ]:
# !pip install langchain

In [7]:
import os
from dotenv import load_dotenv
load=load_dotenv(".env",override=True) # override=True will override the existing env variables with the ones in the .env file, if there are any conflicts. This is useful when we want to test different env variables without having to change the system env variables.

In [6]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen2.5:latest",
    temperature=0.5,
    num_predict=250  # maximum number of tokens the model is allowed to generate in the response
    )

print("start")
response = llm.invoke("What is the capital of France? in one line")
print(response)

start
content='The capital of France is Paris.' additional_kwargs={} response_metadata={'model': 'qwen2.5:latest', 'created_at': '2026-09-08T13:08:05.6308326Z', 'done': True, 'done_reason': 'stop', 'total_duration': 15521823300, 'load_duration': 9911789500, 'prompt_eval_count': 39, 'prompt_eval_duration': 3982786000, 'eval_count': 8, 'eval_duration': 1603021000, 'logprobs': None, 'model_name': 'qwen2.5:latest', 'model_provider': 'ollama'} id='lc_run--01a08121-c0eb-7923-8792-e8f087376127-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 39, 'output_tokens': 8, 'total_tokens': 47}


In [20]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage  
from langchain_core.output_parsers import StrOutputParser 
parser=StrOutputParser()  
response_1=llm.invoke([HumanMessage(content="Hi my name is John.")])
response=llm.invoke([SystemMessage(content="You are a helpful assistant."),
                     AIMessage(content=response_1.content),
             HumanMessage(content="Hey what is my name and What is the capital of France?"),
                
             ])

response.content

'Your name is John, and the capital of France is Paris.'

In [21]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory


store ={}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


with_message_history = RunnableWithMessageHistory(llm, get_session_history)

C:\Users\prashant.saxena\AppData\Local\Temp\ipykernel_10256\1449278162.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
d:\My\self_projects\Langchain and langsmith\venv_langchain_and_langsmith\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [22]:
config ={"configurable": {"session_id": "chat_1"}}


In [24]:
response =with_message_history.invoke([HumanMessage(content="Hi my name is John. I am a data scientist.")], config=config)

In [25]:
response.content

"Hello John! It's great to meet another data scientist. How can I assist you today? Whether you have questions about data analysis, machine learning projects, or need advice on tools and techniques, feel free to let me know. Are there any specific areas or challenges in your work that you'd like to discuss?"

In [26]:
response_2 =with_message_history.invoke([HumanMessage(content="what is my name?")], config=config)
response_2.content

'Your name is John! How can I assist you further, John?'

In [27]:
store["chat_1"]

InMemoryChatMessageHistory(messages=[HumanMessage(content='Hi my name is John. I am a data scientist.', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello John! Nice to meet you. As a data scientist, you must have some exciting projects and challenges that keep you busy. How can I assist you today? Whether it's about data analysis, machine learning models, or any other aspect of your work, feel free to share more details if you'd like advice or just want to discuss something interesting in the field.", additional_kwargs={}, response_metadata={'model': 'qwen2.5:latest', 'created_at': '2026-09-08T14:49:55.2197004Z', 'done': True, 'done_reason': 'stop', 'total_duration': 25305523600, 'load_duration': 11178641600, 'prompt_eval_count': 41, 'prompt_eval_duration': 2528557000, 'eval_count': 75, 'eval_duration': 11576140000, 'logprobs': None, 'model_name': 'qwen2.5:latest', 'model_provider': 'ollama'}, id='lc_run--01a0817e-d444-77f0-9c09-feef4cccae9f-0', tool_calls=[], inval

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder 
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        MessagesPlaceholder(variable_name="messages"),
        
    ]
)

chain = prompt | llm


with_message_history=RunnableWithMessageHistory(
    chain,get_session_history
)
config ={"configurable": {"session_id": "chat_3"}}
response=with_message_history.invoke([HumanMessage(content="my name is prashant.")], config=config)
response.content

AIMessage(content='I\'m sorry, but I don\'t have any information about someone named Prashant in our conversation history. Could you please provide more context or clarify your question? Are you asking about a name, a nickname, or something else related to "Prashant"?', additional_kwargs={}, response_metadata={'model': 'qwen2.5:latest', 'created_at': '2026-09-09T08:59:45.2110545Z', 'done': True, 'done_reason': 'stop', 'total_duration': 12877561500, 'load_duration': 657463500, 'prompt_eval_count': 26, 'prompt_eval_duration': 520833999, 'eval_count': 54, 'eval_duration': 11675073000, 'logprobs': None, 'model_name': 'qwen2.5:latest', 'model_provider': 'ollama'}, id='lc_run--01a08564-ca71-7ce3-b8f9-2747a8db1413-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 26, 'output_tokens': 54, 'total_tokens': 80})

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder 
prompt = ChatPromptTemplate.from_messages(
    [
        ("system",
          "You are a helpful assistant.answer all the question to the best of your ability in {language} language."),
        MessagesPlaceholder(variable_name="messages"),
        
    ]
)

chain = prompt | llm

chain.invoke({"messages": [HumanMessage(content="my name is prashant.")], "language": "Hindi"})



AIMessage(content='नमस्ते, प्रशांत आपका नाम है। कैसे मदद कर सकता हं?', additional_kwargs={}, response_metadata={'model': 'qwen2.5:latest', 'created_at': '2026-09-09T10:44:05.2841195Z', 'done': True, 'done_reason': 'stop', 'total_duration': 18628636600, 'load_duration': 7962222200, 'prompt_eval_count': 39, 'prompt_eval_duration': 1840180000, 'eval_count': 44, 'eval_duration': 8809142000, 'logprobs': None, 'model_name': 'qwen2.5:latest', 'model_provider': 'ollama'}, id='lc_run--01a085c4-3973-7f63-86b5-da77a998f841-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 39, 'output_tokens': 44, 'total_tokens': 83})

In [36]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config ={"configurable": {"session_id": "chat_3"}}
response=with_message_history.invoke({"messages": [HumanMessage(content="what is my name")], "language": "Hindi"}, config=config)
response.content

d:\My\self_projects\Langchain and langsmith\venv_langchain_and_langsmith\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


'आपका नाम प्रशांत है।'